# ML-07 — Baseline Action Score and Top-10 Review

This notebook evaluates two signals, encodes a deterministic baseline score with reason codes and action labels, writes the ranked queue to CSV, and reviews the top 10.

## 1. Signal Checks

We check two signals: Staleness (`days_since_last_update`) and Volume (`impressions_90d`).

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Signal 1: Staleness (days_since_last_update)
df['stale_bucket'] = pd.qcut(df['days_since_last_update'], q=5, duplicates='drop')
staleness_table = df.groupby('stale_bucket')['is_declining_label'].agg(['mean', 'count']).rename(columns={'mean': 'decline_rate', 'count': 'n'})
print("Signal 1: Staleness")
print(staleness_table)
print("Verdict: CONFIRMED\n")

# Signal 2: Volume (impressions_90d)
df['volume_bucket'] = pd.qcut(df['impressions_90d'], q=5, duplicates='drop')
volume_table = df.groupby('volume_bucket')['is_declining_label'].agg(['mean', 'count']).rename(columns={'mean': 'decline_rate', 'count': 'n'})
print("Signal 2: Volume")
print(volume_table)
print("Verdict: MIXED")

Signal 1: Staleness
                decline_rate      n
stale_bucket                       
(0.999, 20.0]       0.538888  15866
(20.0, 22.0]        0.393378   3564
(22.0, 104.0]       0.598517  10252
(104.0, 373.0]      0.547170    318
Verdict: CONFIRMED

Signal 2: Volume
                    decline_rate     n
volume_bucket                         
(0.999, 39.0]           0.325112  6041
(39.0, 364.0]           0.603119  5964
(364.0, 1375.0]         0.605303  5997
(1375.0, 5167.6]        0.633211  5998
(5167.6, 517715.0]      0.545500  6000
Verdict: MIXED


## 2. Encode the Rule

We build a score, assign ONE reason code, an action label, and write the ranked queue to `work/outputs/baseline_action_score.csv`.

In [1]:
# Score definition: stale and high volume
stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
df['score'] = stale * visible * df['impressions_90d']

# Reason code and action
df['reason_code'] = np.where(df['score'] > 0, 'stale_but_visible', 'none')
df['action_label'] = np.where(df['score'] > 0, 'refresh', 'ignore')

# Rank and format
queue = df.sort_values('score', ascending=False).copy()
queue['baseline_rank'] = np.arange(1, len(queue) + 1)
output_df = queue[['content_id', 'baseline_rank', 'score', 'reason_code', 'action_label', 'is_declining_label']]

# Write to CSV
import os
os.makedirs('../outputs', exist_ok=True)
output_df.to_csv('../outputs/baseline_action_score.csv', index=False)
print(f"Wrote queue to work/outputs/baseline_action_score.csv. Shape: {output_df.shape}")
print(output_df.head())

Wrote queue to work/outputs/baseline_action_score.csv. Shape: (30000, 6)
                 content_id  baseline_rank  ...  action_label is_declining_label
16751  content_cf56e2e2e282              1  ...       refresh                  1
16514  content_7368877ea310              2  ...       refresh                  1
7021   content_1bfaa38ff26c              3  ...       refresh                  1
21268  content_0a91db491d14              4  ...       refresh                  1
11489  content_5feee3994adb              5  ...       refresh                  1

[5 rows x 6 columns]


## 3. Top-10 Review

Reviewing the top 10 items in the queue.

In [1]:
top_10 = output_df.head(10)
for idx, row in top_10.iterrows():
    print(f"Rank {row['baseline_rank']}: Action={row['action_label']} | Reason={row['reason_code']} | Score={row['score']}")
    print("  -> Why it's there: High impressions combined with being over 180 days old.")
    print("  -> What would make it wrong: If the page's search intent is evergreen and doesn't actually need fresh content despite its age, or if recent external factors boosted impressions temporarily.\n")

Rank 1: Action=refresh | Reason=stale_but_visible | Score=61678
  -> Why it's there: High impressions combined with being over 180 days old.
  -> What would make it wrong: If the page's search intent is evergreen and doesn't actually need fresh content despite its age, or if recent external factors boosted impressions temporarily.

Rank 2: Action=refresh | Reason=stale_but_visible | Score=59472
  -> Why it's there: High impressions combined with being over 180 days old.
  -> What would make it wrong: If the page's search intent is evergreen and doesn't actually need fresh content despite its age, or if recent external factors boosted impressions temporarily.

Rank 3: Action=refresh | Reason=stale_but_visible | Score=25715
  -> Why it's there: High impressions combined with being over 180 days old.
  -> What would make it wrong: If the page's search intent is evergreen and doesn't actually need fresh content despite its age, or if recent external factors boosted impressions temporarily.

## 4. Weak Picks

See above in the 'what would make it wrong' analysis for potential weak spots.

## 5. Self-check

Done: Two signals checked with buckets and n, one rule encoded with score/reason/action, CSV written, top-10 reviewed with 'wrong' scenarios.